# 스마트 명함/영수증 정리 에이전트 - 단계별 데모

온프레미스 파이프라인을 단계별로 실행해 봅니다.

`OpenCV 전처리 → PaddleOCR → LayoutLM 분류 → Ollama 구조화 → 저장`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# 데모할 이미지 경로 (data/samples/ 에 본인 이미지를 넣으세요)
IMAGE_PATH = '../data/samples/receipt_01.jpg'

## 1) OpenCV 전처리

In [ ]:
import cv2
import matplotlib.pyplot as plt
from src.preprocessing.image_preprocess import preprocess_image

pre = preprocess_image(IMAGE_PATH)
print('debug:', pre.debug)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(cv2.cvtColor(pre.original, cv2.COLOR_BGR2RGB)); ax[0].set_title('original')
ax[1].imshow(cv2.cvtColor(pre.processed, cv2.COLOR_BGR2RGB)); ax[1].set_title('processed')
plt.show()

## 2) PaddleOCR 텍스트 추출

In [ ]:
from src.ocr.paddle_ocr_engine import run_ocr

ocr = run_ocr(pre.processed)
print('words:', len(ocr.words), '| mean_conf:', round(ocr.mean_confidence, 3))
print(ocr.full_text)

## 3) LayoutLM 문서 분류 (가중치 없으면 키워드 fallback)

In [ ]:
from src.classification.layoutlm_classifier import classify_document

cls = classify_document(ocr)
print(cls)

## 4) Ollama 구조화 (미연결 시 정규식 fallback)

In [ ]:
from src.schemas import DocumentType
from src.extraction.ollama_extractor import extract_receipt, extract_business_card

if cls.doc_type == DocumentType.RECEIPT:
    print(extract_receipt(ocr.full_text).model_dump())
elif cls.doc_type == DocumentType.BUSINESS_CARD:
    print(extract_business_card(ocr.full_text).model_dump())
else:
    print('미상 문서')

## 5) end-to-end 에이전트 + Excel 저장

In [ ]:
from src.pipeline.agent import DocumentAgent
from src.storage.excel_exporter import export_to_excel

agent = DocumentAgent(persist=True)
result, _ = agent.process_image(IMAGE_PATH)
print('type:', result.classification.doc_type, '| structured:', result.structured_dict())

path = export_to_excel([result])
print('saved:', path)